In [ ]:
!pip install -U llmcompressor

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import hf_hub_download, HfApi, login
from datasets import Dataset
import os
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from google.colab import userdata
import textwrap

# ── Config ───────────────────────────────────────────────────────────────────
BASE_MODEL      = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_REPO    = "pshashid/llama3.1B_SQL_Finetuned"
TARGET_REPO     = "pshashid/llama3.1B_8B_SQL_Finetuned_model"
MERGED_DIR      = "./merged_model"
QUANTIZED_DIR   = "./quantized_model"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

MERGED_DIR    = "./merged_model"
QUANTIZED_DIR = "./quantized_model"
MAX_SEQ_LEN   = 2048
NUM_SAMPLES   = 64   # increase to 512 for better quality, slower run





In [ ]:
# quantize_and_push.py
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1: Merge LoRA → Full Model
# STAGE 2: Quantize to NVFP4 (weights) + FP8 KV-Cache
# STAGE 3: Push to HuggingFace
#
# Requirements:
#   pip install llmcompressor transformers peft huggingface_hub torch
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1: Merge LoRA adapter into base weights
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STAGE 1: Merging LoRA adapter into base model...")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Pull your custom chat template
template_path = hf_hub_download(repo_id=ADAPTER_REPO, filename="chat_template.jinja")
with open(template_path, 'r') as f:
    tokenizer.chat_template = f.read()

tokenizer.pad_token    = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

# Load base + adapter, then merge
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype = torch.float16,
    device_map  = "auto"
)
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)

print("  Merging weights (this fuses LoRA A/B matrices into base)...")
merged_model = peft_model.merge_and_unload()  # Returns a plain HF model
merged_model.eval()

# Save merged model locally
os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"  Merged model saved to: {MERGED_DIR}")

STAGE 1: Merging LoRA adapter into base model...


chat_template.jinja:   0%|          | 0.00/529 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

  Merging weights (this fuses LoRA A/B matrices into base)...
  Merged model saved to: ./merged_model


In [ ]:
# quantize_and_push.py
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1: Merge LoRA → Full Model
# STAGE 2: Quantize to NVFP4 (weights) + FP8 KV-Cache
# STAGE 3: Push to HuggingFace
#
# Requirements:
#   pip install llmcompressor transformers peft huggingface_hub torch
# ─────────────────────────────────────────────────────────────────────────────


# ─────────────────────────────────────────────────────────────────────────────
# STAGE 1: Merge LoRA adapter into base weights
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("STAGE 1: Merging LoRA adapter into base model...")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Pull your custom chat template
template_path = hf_hub_download(repo_id=ADAPTER_REPO, filename="chat_template.jinja")
with open(template_path, 'r') as f:
    tokenizer.chat_template = f.read()

tokenizer.pad_token    = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

# Load base + adapter, then merge
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype = torch.float16,
    device_map  = "auto"
)
peft_model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)

print("  Merging weights (this fuses LoRA A/B matrices into base)...")
merged_model = peft_model.merge_and_unload()  # Returns a plain HF model
merged_model.eval()

# Save merged model locally
os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"  Merged model saved to: {MERGED_DIR}")



STAGE 1: Merging LoRA adapter into base model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  Merging weights (this fuses LoRA A/B matrices into base)...
  ✅ Merged model saved to: ./merged_model


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 2 (FINAL - VERIFIED FROM OFFICIAL DOCS): NVFP4 + FP8 KV-Cache
# Replace your entire Stage 2 cell with this
# ─────────────────────────────────────────────────────────────────────────────


# ── Load tokenizer + model ───────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype = torch.float16,
    device_map  = "auto",
)

# ── Build calibration dataset ────────────────────────────────────────────────
SQL_SAMPLES = [
    ("Table: orders(order_id, customer_id, amount, created_at)",
     "Get total sales per customer for the last 30 days, ordered by highest spend.",
     "SELECT customer_id, SUM(amount) FROM orders WHERE created_at >= NOW() - INTERVAL 30 DAY GROUP BY customer_id ORDER BY SUM(amount) DESC;"),
    ("Tables: employees(emp_id, name, dept_id, salary), departments(dept_id, dept_name)",
     "Find average salary per department with more than 5 employees.",
     "SELECT d.dept_name, AVG(e.salary) FROM employees e JOIN departments d ON e.dept_id = d.dept_id GROUP BY d.dept_name HAVING COUNT(e.emp_id) > 5;"),
    ("Table: products(product_id, name, category, price, stock_quantity)",
     "Which categories have more than 100 units in stock?",
     "SELECT category, SUM(stock_quantity) FROM products GROUP BY category HAVING SUM(stock_quantity) > 100;"),
    ("Table: sessions(session_id, user_id, started_at, ended_at, page_views)",
     "Find users with more than 10 sessions in the last 7 days.",
     "SELECT user_id, COUNT(*) as session_count FROM sessions WHERE started_at >= NOW() - INTERVAL 7 DAY GROUP BY user_id HAVING COUNT(*) > 10;"),
    ("Tables: invoices(invoice_id, client_id, total, paid, due_date), clients(client_id, name, country)",
     "List overdue unpaid invoices by country.",
     "SELECT c.country, COUNT(*) as overdue_count, SUM(i.total) as total_owed FROM invoices i JOIN clients c ON i.client_id = c.client_id WHERE i.paid = FALSE AND i.due_date < NOW() GROUP BY c.country ORDER BY total_owed DESC;"),
]

# Repeat samples to hit NUM_SAMPLES
raw_texts = []
for schema, query, sql in SQL_SAMPLES * (NUM_SAMPLES // len(SQL_SAMPLES) + 1):
    messages = [
        {"role": "system",    "content": f"You are an expert SQL engineer. Return only valid SQL. Schema: {schema}"},
        {"role": "user",      "content": query},
        {"role": "assistant", "content": sql},
    ]
    raw_texts.append(tokenizer.apply_chat_template(messages, tokenize=False))

raw_texts = raw_texts[:NUM_SAMPLES]

# Tokenize into HF Dataset (required by oneshot — plain lists don't work)
def tokenize(sample):
    return tokenizer(
        sample["text"],
        padding        = False,
        max_length     = MAX_SEQ_LEN,
        truncation     = True,
        add_special_tokens = False,
    )

ds = Dataset.from_dict({"text": raw_texts})
ds = ds.map(tokenize, remove_columns=ds.column_names)   # ← removes "text" col, keeps input_ids

# ── Recipe: NVFP4 weights + FP8 KV-Cache ────────────────────────────────────
# scheme="NVFP4"           — correct preset string (not "FP4")
# kv_cache_scheme as dict  — correct format per official docs
# num_calibration_samples  — in oneshot(), NOT in QuantizationModifier()
recipe = QuantizationModifier(
    targets         = "Linear",
    scheme          = "NVFP4",          # Blackwell native FP4
    ignore          = ["lm_head"],
    kv_cache_scheme = {                 # Dict form (also accepted by pydantic)
        "num_bits"  : 8,
        "type"      : "float",
        "strategy"  : "tensor",
        "dynamic"   : False,
        "symmetric" : True,
    },
)

print("Running NVFP4 calibration + quantization (~10–20 min)...")

oneshot(
    model                   = model,
    dataset                 = ds,           # HF Dataset, not a list
    recipe                  = recipe,
    max_seq_length          = MAX_SEQ_LEN,
    num_calibration_samples = NUM_SAMPLES,  # Correct location
)

os.makedirs(QUANTIZED_DIR, exist_ok=True)
model.save_pretrained(QUANTIZED_DIR, save_compressed=True)
tokenizer.save_pretrained(QUANTIZED_DIR)

print(f"Done! Quantized model saved to: {QUANTIZED_DIR}")
print(f"Check size: expected ~4.5GB")

The tokenizer you are loading from './merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Map:   0%|          | 0/64 [00:00<?, ? examples/s]

Running NVFP4 calibration + quantization (~10–20 min)...


The tokenizer you are loading from './merged_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


...

W0302 21:35:24.213000 4954 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(33/33): Propagating: 100%|██████████| 64/64 [00:00<00:00, 1530.07it/s]

2026-03-02T21:37:12.422834+0000 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-03-02T21:37:12.423448+0000 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
2026-03-02T21:37:12.460991+0000 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.



Compressing model: 224it [02:16,  1.64it/s]
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3970: UserWarning: Attempting to save a model with offloaded modules. Ensure that unallocated cpu memory exceeds the `shard_size` (5GB default)
  warnings.warn(


Saving checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Done! Quantized model saved to: ./quantized_model
Check size: expected ~4.5GB


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# STAGE 3: Push Quantized Model to HuggingFace
# ─────────────────────────────────────────────────────────────────────────────

# ── Login (uses your HF token) ───────────────────────────────────────────────
# Option A: interactive prompt
login(token=userdata.get("HF_TOKEN_WRITE"))
# Option B: token directly (if running headless/automated)
# login(token="hf_your_token_here")

# ── Write Model Card ─────────────────────────────────────────────────────────
MODEL_CARD = f"""---
base_model: meta-llama/Llama-3.1-8B-Instruct
tags:
  - sql
  - llama
  - nvfp4
  - quantized
  - vllm
  - blackwell
  - llmcompressor
library_name: transformers
pipeline_tag: text-generation
---

# Llama 3.1 8B SQL — NVFP4 Quantized (Blackwell)

SQL generation model fine-tuned on text-to-SQL tasks, quantized for NVIDIA Blackwell (RTX 50-series) using `llm-compressor`.

## Quantization Details

| Component | Format   | Notes                                      |
|-----------|----------|--------------------------------------------|
| Weights   | NVFP4    | ~4.5GB — Blackwell 5th-gen Tensor Core native |
| KV-Cache  | FP8      | 50% memory vs FP16 — configured via vLLM  |
| Activations | FP16   | `lm_head` kept in FP16 for output quality |

## vLLM Inference (RTX 5090)
```bash
vllm serve {TARGET_REPO} \\
  --dtype float16 \\
  --quantization fp4 \\
  --kv-cache-dtype fp8 \\
  --max-model-len 131072 \\
  --gpu-memory-utilization 0.85 \\
  --enable-prefix-caching \\
  --port 8000
```

## Performance Targets (Dual RTX 5090 Pod — 8 Replicas)

| Metric                  | Target        |
|-------------------------|---------------|
| Time to First Token     | < 15ms        |
| Throughput (1 replica)  | ~200 tok/s    |
| Aggregate (8 replicas)  | 1,500+ tok/s  |
| Max Concurrency         | 100+ users    |

## Example Usage (Python)
```python
from vllm import LLM, SamplingParams

llm = LLM(
    model                  = "{TARGET_REPO}",
    quantization           = "fp4",
    kv_cache_dtype         = "fp8",
    max_model_len          = 131072,
    enable_prefix_caching  = True,
)

sampling = SamplingParams(temperature=0, max_tokens=200)
outputs  = llm.generate(["SELECT"], sampling)
print(outputs[0].outputs[0].text)
```
"""

with open(f"{QUANTIZED_DIR}/README.md", "w") as f:
    f.write(MODEL_CARD)

print("✅ Model card written.")



✅ Model card written.


In [ ]:
token = userdata.get("HF_TOKEN_WRITE")
api = HfApi(token=token)  # ← pass token directly to the API instance

# Verify it's a write token
whoami = api.whoami()
print(f"Logged in as: {whoami['name']}")
print(f"Token type: {whoami['auth']['accessToken']['role']}")  # should print "write"

# Create repo
api.create_repo(
    repo_id  = TARGET_REPO,
    exist_ok = True,
    private  = False,
    token    = token,   # ← explicit here too
)

# Upload
api.upload_folder(
    folder_path    = QUANTIZED_DIR,
    repo_id        = TARGET_REPO,
    repo_type      = "model",
    token          = token,   # ← and here
    commit_message = "Upload NVFP4 quantized model with FP8 KV-cache",
)

print(f"🚀 Done! https://huggingface.co/{TARGET_REPO}")

Logged in as: pshashid
Token type: write


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   3%|3         | 32.0MB / 1.05GB            

  ...ized_model/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...0001-of-00002.safetensors:   0%|          | 24.0MB / 4.98GB            

🚀 Done! https://huggingface.co/pshashid/llama3.1B_8B_SQL_Finetuned_model


In [ ]:
MODEL_REPO = "pshashid/llama3.1B_8B_SQL_Finetuned_model"

# ── Check hardware first ─────────────────────────────────────────────────────
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO)

# Load in bfloat16 — matches what compressed_tensors decompresses weights into
model = AutoModelForCausalLM.from_pretrained(
    MODEL_REPO,
    torch_dtype = torch.bfloat16,  # ← MUST be bfloat16, not float16
    device_map  = "auto",
)
model.eval()
print(f"Device : {next(model.parameters()).device}")
print(f"Dtype  : {next(model.parameters()).dtype}")  # should print bfloat16

# ── Inference ────────────────────────────────────────────────────────────────
def run_sql_inference(schema, query, max_new_tokens=200):
    messages = [
        {"role": "system",  "content": f"You are an expert SQL engineer. Return only valid SQL. Schema context: {schema}"},
        {"role": "user",    "content": query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Cast inputs to bfloat16-compatible device (no dtype cast needed for input_ids — they're int)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# ── Tests ────────────────────────────────────────────────────────────────────
SQL_TEST_CASES = [
    {"schema": "Table: orders(order_id, customer_id, amount, created_at)",
     "query":  "Get total sales per customer for the last 30 days, ordered by highest spend."},
    {"schema": "Tables: employees(emp_id, name, dept_id, salary), departments(dept_id, dept_name)",
     "query":  "Find average salary per department, only for departments with more than 5 employees."},
    {"schema": "Table: products(product_id, name, category, price, stock_quantity)",
     "query":  "Which product categories have more than 100 total units in stock?"},
]

print("\n" + "=" * 70)
for i, tc in enumerate(SQL_TEST_CASES, 1):
    print(f"\n[Test {i}]")
    print(f"  Schema : {tc['schema']}")
    print(f"  Query  : {tc['query']}")
    result = run_sql_inference(tc["schema"], tc["query"])
    print(f"  Output :\n{textwrap.indent(result, '    ')}")
    print("-" * 70)

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
BF16 support: True


`torch_dtype` is deprecated! Use `dtype` instead!
Compressing model: 224it [00:00, 1722.95it/s]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Device : cuda:0
Dtype  : torch.bfloat16


[Test 1]
  Schema : Table: orders(order_id, customer_id, amount, created_at)
  Query  : Get total sales per customer for the last 30 days, ordered by highest spend.
  Output :
    Here is the SQL query to get the total sales per customer for the last 30 days, ordered by highest spend:

    ```sql
    SELECT customer_id, SUM(amount) as total_sales
    FROM orders
    WHERE created_at > (CURRENT_DATE - INTERVAL '30' DAY)
    GROUP BY customer_id
    ORDER BY total_sales DESC;
    ```

    This query:

    1. Selects the `customer_id` and calculates the `SUM` of `amount` for each group of customers.
    2. Filters the results to only include orders created in the last 30 days.
    3. Groups the results by `customer_id`.
    4. Orders the results by the `total_sales` in descending order (highest spend first).
----------------------------------------------------------------------

[Test 2]
  Schema : Tables: employees(emp_id, name, dept_id, salary),